<h1>Búsquedas no informadas</h1></th></tr></tbody></table>

Este cuaderno contiene problemas clásicos sobre búsquedas no informadas

In [1]:
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.optimize as opt
import time
from search import *

#### 1. Se dispone de dos cántaros de agua, uno de 4 litros y otro de 3 litros de capacidad, siendo esta la única información que se tiene de los mismos. Existe una bomba de agua con la que se pueden llenar los cántaros. Se desea que el cántaro de 4 litros de capacidad quede lleno por la mitad y el de 3 litros vacío.


In [2]:
class Cantaros(Problem):
    def __init__(self, initial=(0,0),goal = (2,0)):#suponemos que de entrada ambos estan vacios
        super().__init__(initial,goal)
    
    def actions(self,state):
        c4,c3 = state
        possible_actions = []
        if c4 < 4:
            possible_actions.append("LLENAR_C4")
        if c3 < 3:
            possible_actions.append("LLENAR_C3")
        if c4 > 0:
            possible_actions.append("VACIAR_C4")
        if c3 > 0:
            possible_actions.append("VACIAR_C3")
        if c4 > 0 and c3 < 3:
            possible_actions.append("TRASVASAR_4_3")
        if c3 > 0 and c4 < 4:
              possible_actions.append("TRASVASAR_3_4")
        return possible_actions
    
    def result(self,state,action):
        c4,c3 = state

        if action == "LLENAR_C4":
            return(4,c3)
        elif action == "LLENAR_C3":
            return(c4,3)
        elif action == "VACIAR_C4":
            return (0,c3)
        elif action == "VACIAR_C3":
            return (c4,0)
        elif action == "TRASVASAR_4_3":
            espacio = 3- c3
            cantidad = min(c4,espacio)
            return (c4 - cantidad, c3 + cantidad)
        elif action == "TRASVASAR_3_4":
            espacio = 4 - c4
            cantidad = min(espacio,c3)
            return (c4+ cantidad, c3 - cantidad)
    def test_goal(self,state):
        return self.goal == state
    

In [3]:
star_time = time.time()
c = Cantaros()

print('Problema de los Cantaros.')
print(c.initial)
print(c.goal)
print(c.actions((0,0)))
print(c.goal_test((2,0)))

sol_c = breadth_first_tree_search(c)
print("Tiempo transcurrido (breadth_first_tree_search): ",time.time() - star_time)
print("Movimientos cantaros:",sol_c.solution())
print("Solucion P:", sol_c)

Problema de los Cantaros.
(0, 0)
(2, 0)
['LLENAR_C4', 'LLENAR_C3']
True
Tiempo transcurrido (breadth_first_tree_search):  0.0
Movimientos cantaros: ['LLENAR_C3', 'TRASVASAR_3_4', 'LLENAR_C3', 'TRASVASAR_3_4', 'VACIAR_C4', 'TRASVASAR_3_4']
Solucion P: <Node (2, 0)>


#### 2. Hace mucho tiempo un granjero fue al mercado y compró un lobo, una cabra y una col. Para volver a su casa tenía que cruzar un río. El granjero dispone de una barca para cruzar a la otra orilla, pero en la barca solo caben él y una de sus compras. Si el lobo se queda solo con la cabra se la come, y si la cabra se queda sola con la col se la come. El reto del granjero era cruzar él mismo y dejar sus compras a la otra orilla del río, dejando cada compra intacta. ¿Cómo lo hizo? Representar el árbol de búsqueda con la herramienta “Search” de AIspace, implementar en Python la solución utilizando AIMA y realizar una comparativa de los distintos algoritmos de búsqueda no informada.

In [4]:
class Granjero(Problem):
    def __init__(self, initial = (1,1,1,1,1), goal = (0,0,0,0,0)):
        super().__init__(initial,goal)

    def getCol(self, state):
        return state[0]
    def getCabra(self, state):
        return state[1]
    def getLobo(self, state):
        return state[2]
    def getGranjero(self, state):
        return state[3]
    def getBarca(self, state):
        return state[4]

    def estadoPeligroso(self, c, cabra, l, g, b):
            #orilla izquierda                                                                                              #orilla derecha
        return(((l == 1 and cabra == 1) and (g == 0 and c == 0))) or (((c == 1 and cabra == 1) and (g==0 and l == 0))) or ((l == cabra and (g != 0 and c != 0)) or (c == cabra and (g !=0 and l != 0)))
    def canMoveBoat(self,state, where):
        col = self.getCol(state)
        cabra = self.getCabra(state)
        l = self.getLobo(state)
        g = self.getGranjero(state)
        b = self.getBarca(state)

        if where == 'GRANJERO':
            if b == 1:
                return g == 1 and not self.estadoPeligroso(col,cabra,l, g - 1 ,b)
            else:
                return g == 0 and not self.estadoPeligroso(col,cabra,l, g + 1 , b)

        elif where == 'GRANJEROCABRA':
            if b == 1:
                return g == 1 and cabra == 1 and not self.estadoPeligroso(col,cabra -1, l, g - 1 ,b)
            else:
                return g == 0 and cabra == 0 and not self.estadoPeligroso(col,cabra + 1, l, g + 1 , b)
        elif where == 'GRANJEROCOL':
            
            if b == 1:
                return g == 1 and col == 1 and not self.estadoPeligroso(col -1 ,cabra, l, g - 1 ,b)
            else:
                return g == 0 and col == 0 and not self.estadoPeligroso(col + 1,cabra, l, g + 1 , b)
        elif where == 'GRANJEROLOBO':
            if b == 1:
                return g == 1 and l == 1 and not self.estadoPeligroso(col,cabra , l -1 , g - 1 ,b)
            else:
                return g == 0 and l == 0 and not self.estadoPeligroso(col,cabra , l + 1, g + 1 , b)
        return False        


    def actions(self, state):
        possible_actions = []

        for move in ['GRANJERO', 'GRANJEROCABRA', 'GRANJEROCOL','GRANJEROLOBO']:
            if self.canMoveBoat(state, move):
                possible_actions.append(move)
        return possible_actions

    def result(self, state,action):
        new_state = list(state)
        deltas = { 'GRANJEROCOL':(1,0,0,1),'GRANJEROCABRA':(0,1,0,1), 'GRANJEROLOBO':(0,0,1,1),'GRANJERO': (0,0,0,1)}
        dcol,dcabra,dl,dg = deltas[action]

        barca = self.getBarca(state)
        if barca == 1:
            new_state[0] -= dcol
            new_state[1] -= dcabra
            new_state[2] -= dl
            new_state[3] -= dg
            new_state[4] = 0
        else:
            new_state[0] += dcol
            new_state[1] += dcabra
            new_state[2] += dl
            new_state[3] += dg
            new_state[4] = 1

        return tuple(new_state)
    def goal_test(self, state):
        return state == self.goal

In [4]:
star_time = time.time()
g = Granjero()

print('Problema del Granjero.')



sol_g = breadth_first_tree_search(g)
print("Tiemo transcurrido (breadth_first_tree_search): ",time.time() - star_time)
print("Movimientos Ganjero:",sol_g.solution())
print("Solucion P:", sol_g)

Problema del Granjero.
Tiemo transcurrido (breadth_first_tree_search):  0.0
Movimientos Ganjero: ['GRANJEROCABRA', 'GRANJERO', 'GRANJEROCOL', 'GRANJEROCABRA', 'GRANJEROLOBO', 'GRANJERO', 'GRANJEROCABRA']
Solucion P: <Node (0, 0, 0, 0, 0)>
